In [3]:
import pandas as pd

df = pd.read_csv('../data/raw/santander-customer-transaction-prediction/train.csv')
X = df.drop(["target", "ID_code"], axis=1)
y = df["target"]

df_test = pd.read_csv('../data/raw/santander-customer-transaction-prediction/test.csv')
X_test = df_test.drop("ID_code", axis=1)

In [4]:
X_fe = X.copy()

X_fe["row_mean"] = X.mean(axis=1)
X_fe["row_std"] = X.std(axis=1)
X_fe["row_min"] = X.min(axis=1)
X_fe["row_max"] = X.max(axis=1)

In [5]:
X_test_fe = X_test.copy()

X_test_fe["row_mean"] = X_test.mean(axis=1)
X_test_fe["row_std"] = X_test.std(axis=1)
X_test_fe["row_min"] = X_test.min(axis=1)
X_test_fe["row_max"] = X_test.max(axis=1)

In [6]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_auc_scores = []
fold_predictions = np.zeros(len(df))
test_preds = np.zeros(len(X_test_fe))
oof_preds = np.zeros(len(X_fe))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_fe, y)):
    X_train_fold, X_val_fold = X_fe.iloc[train_idx], X_fe.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        random_state=42
    )
    
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),
            lgb.log_evaluation(period=100)
        ]
    )
    
    fold_preds = model.predict_proba(X_val_fold)[:, 1]
    fold_auc = roc_auc_score(y_val_fold, fold_preds)
    fold_auc_scores.append(fold_auc)

    oof_preds[val_idx] = fold_preds
    test_preds += (
        model.predict_proba(
            X_test_fe,
            num_iteration=model.best_iteration_
        )[:, 1]
        / skf.n_splits
    )
    
    print(f"Fold {fold + 1} AUC: {fold_auc:.6f}")
    oof_auc = roc_auc_score(y, oof_preds)

print(f"\nMean AUC: {np.mean(fold_auc_scores):.6f}")
print(f"Std AUC: {np.std(fold_auc_scores):.6f}")
print(f"OOF AUC: {oof_auc:.6f}")

[LightGBM] [Info] Number of positive: 16079, number of negative: 143921
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.066583 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 52020
[LightGBM] [Info] Number of data points in the train set: 160000, number of used features: 204
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100494 -> initscore=-2.191750
[LightGBM] [Info] Start training from score -2.191750
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.84598	valid_0's binary_logloss: 0.260648
[200]	valid_0's auc: 0.870798	valid_0's binary_logloss: 0.238938
[300]	valid_0's auc: 0.8817	valid_0's binary_logloss: 0.227169
[400]	valid_0's auc: 0.887309	valid_0's binary_logloss: 0.219845
[500]	valid_0's auc: 0.890472	valid_0's binary_logloss: 0.215252
[600]	valid_0's auc: 0.892308	valid_0's binary_logloss: 0.212337
[700]	valid_0's auc: 0.893244	valid_0's binary_loglos

In [ ]:
'''
Baseline AUC in exp 2: 0.891594
OOF AUC in exp 3: 0.891727

Improvement: 0.000133 << Std dev in AUC across folds: 0.00314

No idea about whether the row transformation helped or not.
'''

In [7]:
submission = pd.read_csv("../data/raw/santander-customer-transaction-prediction/sample_submission.csv")
submission["target"] = test_preds
submission.to_csv("../submissions/exp003.csv", index=False)